# 🎤 ScribeCrate Live Real-Time Transcription | Google Colab Demo

This notebook runs the **[ScribeCrate Live](https://github.com/hwdsl2/scribecrate-live)** server directly in Colab (without Docker).

The server provides two interfaces powered by [WhisperLive](https://github.com/collabora/WhisperLive) and [faster-whisper](https://github.com/SYSTRAN/faster-whisper):

- **WebSocket streaming** — real-time transcription as audio is streamed, with segments printed as they are decoded
- **OpenAI-compatible REST API** — standard `POST /v1/audio/transcriptions` for file transcription

**Quick start:** Select **Runtime → Run all** to run the full demo automatically using a built-in sample audio file.  
**Runtime recommendation:** GPU is used automatically when available. CPU works too.

## Step 1 — Install dependencies

In [ ]:
import subprocess, sys

def pip_install(*packages):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet",
         "--disable-pip-version-check", *packages],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        raise RuntimeError("Dependency installation failed. See pip output above.")

# Keep the server and WebSocket library aligned with the Docker images.
# Install only the dependencies needed for this faster-whisper demo.
pip_install("whisper-live==0.10.0", "--no-deps")
pip_install(
    "faster-whisper==1.2.0", "av>=11,<19",  # PyAV 19 removed metadata_errors.
    "websockets==17.1", "onnxruntime>=1.20.0,<2", "kaldialign",
    "websocket-client", "fastapi", "uvicorn", "python-multipart",
    "numpy>=1.26.4,<2.5", "soundfile", "requests",
)

try:
    _install_has_gpu = subprocess.run(
        ["nvidia-smi"], capture_output=True, timeout=5,
    ).returncode == 0
except (OSError, subprocess.TimeoutExpired):
    _install_has_gpu = False

# CTranslate2 needs CUDA 12 and cuDNN 9 even if Colab's PyTorch uses CUDA 13.
# Leave Colab's preinstalled PyTorch unchanged and skip GPU wheels on CPU.
if _install_has_gpu:
    pip_install("nvidia-cublas-cu12", "nvidia-cudnn-cu12>=9,<10")

print('✅ Installation complete.')

## Step 2 — Configure model & runtime settings

Edit the variables below to choose a different model or language.

The notebook defaults to multilingual `small`; Docker defaults to `base`.

After changing models, rerun Steps 2 and 3.

Set `WHISPERLIVE_API_KEY` to require a key for transcription, or leave it blank to disable authentication. `/docs`, `/redoc`, `/openapi.json` and `/health` remain public.

| Model | Disk | RAM | Notes |
|---|---|---|---|
| `tiny` | ~75 MB | ~250 MB | Fastest, lower accuracy |
| `base` | ~145 MB | ~700 MB | Good balance; Docker default |
| `small` | ~465 MB | ~1.5 GB | Better accuracy; **notebook default** |
| `medium` | ~1.5 GB | ~5 GB | High accuracy |
| `large-v3-turbo` | ~1.6 GB | ~6 GB | Fast + high accuracy ⭐; transcription only |
| `large-v3` | ~3 GB | ~10 GB | Best accuracy |

Memory use varies with hardware and precision. A GPU is recommended for larger models.

In [ ]:
import os, subprocess

# ── Model settings ──────────────────────────────────────────────────────────
WHISPERLIVE_MODEL    = "small"  # tiny | base | small | medium | large-v3-turbo | large-v3
WHISPERLIVE_LANGUAGE = "auto"  # auto, or a BCP-47 code e.g. "en", "fr", "zh"

# ── Server settings ─────────────────────────────────────────────────────────
WHISPERLIVE_PORT      = 9091   # WebSocket port
WHISPERLIVE_REST_PORT = 8001   # REST API port
WHISPERLIVE_API_KEY   = ""     # optional API key for this local Colab server

# ── Runtime detection ───────────────────────────────────────────────────────
try:
    has_gpu = subprocess.run(
        ["nvidia-smi"], capture_output=True, timeout=5,
    ).returncode == 0
except (OSError, subprocess.TimeoutExpired):
    has_gpu = False

print(f"GPU available : {has_gpu}")
print(f"Model         : {WHISPERLIVE_MODEL}")
print(f"Language      : {WHISPERLIVE_LANGUAGE}")
print(f"WebSocket port: {WHISPERLIVE_PORT}")
print(f"REST API port : {WHISPERLIVE_REST_PORT}")
print('✅ Configuration set.')

## Step 3 — Start the server

In [ ]:
import os, subprocess, sys, time, urllib.error, urllib.request, textwrap
from importlib.metadata import distribution

def stop_server():
    if "server_proc" in globals() and server_proc.poll() is None:
        server_proc.terminate()
        try:
            server_proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            server_proc.kill()
            server_proc.wait()

# Stop any previously running server instance before starting a new one.
stop_server()

# Map model names to HuggingFace repo IDs.
# faster-whisper accepts HuggingFace repo IDs (org/repo) directly;
# the path must contain '/' to bypass upstream path validation.
_MODEL_TO_HF = {
    "tiny":             "Systran/faster-whisper-tiny",
    "tiny.en":          "Systran/faster-whisper-tiny.en",
    "base":             "Systran/faster-whisper-base",
    "base.en":          "Systran/faster-whisper-base.en",
    "small":            "Systran/faster-whisper-small",
    "small.en":         "Systran/faster-whisper-small.en",
    "medium":           "Systran/faster-whisper-medium",
    "medium.en":        "Systran/faster-whisper-medium.en",
    "large-v1":         "Systran/faster-whisper-large-v1",
    "large-v2":         "Systran/faster-whisper-large-v2",
    "large-v3":         "Systran/faster-whisper-large-v3",
    "large-v3-turbo":   "mobiuslabsgmbh/faster-whisper-large-v3-turbo",
    "turbo":            "mobiuslabsgmbh/faster-whisper-large-v3-turbo",
}
hf_repo = _MODEL_TO_HF.get(WHISPERLIVE_MODEL, f"Systran/faster-whisper-{WHISPERLIVE_MODEL}")

# Build a small inline launcher script that starts TranscriptionServer.
launcher = textwrap.dedent(f"""\
    import os, logging
    if {has_gpu!r}:
        import ctypes
        ctypes.CDLL("libcublas.so.12")
        ctypes.CDLL("libcudnn.so.9")
    from whisper_live.server import TranscriptionServer
    logging.root.setLevel(logging.INFO)
    os.makedirs("/content/whisper_live_cache", exist_ok=True)
    os.environ["HF_HOME"] = "/content/whisper_live_cache"
    os.environ["HF_HUB_CACHE"] = "/content/whisper_live_cache"
    server = TranscriptionServer()
    server.run(
        "127.0.0.1",
        port={WHISPERLIVE_PORT},
        backend="faster_whisper",
        faster_whisper_custom_model_path="{hf_repo}",
        max_clients=4,
        max_connection_time=600,
        cache_path="/content/whisper_live_cache",
        rest_port={WHISPERLIVE_REST_PORT},
        enable_rest=True,
        api_key=os.environ.get("WHISPERLIVE_API_KEY") or None,
    )
""")

launcher_path = "/content/whisper_live_server.py"
with open(launcher_path, "w") as f:
    f.write(launcher)

server_env = os.environ.copy()
server_env["WHISPERLIVE_API_KEY"] = WHISPERLIVE_API_KEY
if has_gpu:
    # Set library paths before starting Python, as required by CTranslate2.
    cuda_lib_dirs = [
        str(distribution("nvidia-cublas-cu12").locate_file("nvidia/cublas/lib")),
        str(distribution("nvidia-cudnn-cu12").locate_file("nvidia/cudnn/lib")),
    ]
    existing_lib_path = server_env.get("LD_LIBRARY_PATH")
    if existing_lib_path:
        cuda_lib_dirs.append(existing_lib_path)
    server_env["LD_LIBRARY_PATH"] = os.pathsep.join(cuda_lib_dirs)
with open("/content/whisper_live_server.log", "w") as log_file:
    server_proc = subprocess.Popen(
        [sys.executable, launcher_path],
        env=server_env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

try:
    # /docs is public and checks readiness, not authentication.
    # The model is loaded lazily during the first transcription.
    docs_url = f"http://127.0.0.1:{WHISPERLIVE_REST_PORT}/docs"
    local_http = urllib.request.build_opener(urllib.request.ProxyHandler({}))
    print(f"Starting server (pid {server_proc.pid}), waiting for REST API to be ready…")

    ready = False
    deadline = time.monotonic() + 120  # Allow about two minutes for startup.
    while time.monotonic() < deadline:
        if server_proc.poll() is not None:
            break
        try:
            with local_http.open(docs_url, timeout=3) as r:
                if r.status == 200:
                    ready = True
                    break
        except (urllib.error.URLError, TimeoutError):
            pass
        print(".", end="", flush=True)
        time.sleep(2)

    if not ready:
        print("\n❌ Server exited or did not become ready in time. Check logs:")
        with open("/content/whisper_live_server.log") as _f:
            log_tail = _f.read()[-3000:]
        if WHISPERLIVE_API_KEY:
            log_tail = log_tail.replace(WHISPERLIVE_API_KEY, "[REDACTED]")
        print(log_tail)
        raise RuntimeError("Server startup failed. See server logs above.")

    # An empty POST checks authentication without downloading a model: valid
    # credentials (or disabled auth) reach file validation and return HTTP 422.
    auth_cases = [({}, 422)]
    if WHISPERLIVE_API_KEY:
        auth_cases = [
            ({}, 401),
            ({"Authorization": f"Bearer {WHISPERLIVE_API_KEY}-invalid"}, 401),
            ({"Authorization": f"Bearer {WHISPERLIVE_API_KEY}"}, 422),
        ]
    for headers, expected_status in auth_cases:
        req = urllib.request.Request(
            f"http://127.0.0.1:{WHISPERLIVE_REST_PORT}/v1/audio/transcriptions",
            data=b"", headers=headers, method="POST",
        )
        try:
            with local_http.open(req, timeout=5) as r:
                status = r.status
        except urllib.error.HTTPError as exc:
            status = exc.code
            exc.close()
        if status != expected_status:
            raise RuntimeError(
                f"Authentication check failed: expected HTTP {expected_status}, got {status}."
            )

    auth_status = "enabled and verified" if WHISPERLIVE_API_KEY else "disabled (no key configured)"
    print(f"\n✅ Server ready. REST transcription authentication: {auth_status}.")
except BaseException:
    # Also clean up on a failed check or an interrupted startup cell.
    stop_server()
    raise

## Step 4 — Download sample audio

The cell below downloads a sample audio file automatically so you can try the API right away.  
To use your own audio instead, uncomment and run the **Upload your own file** cell, then re-run the **Set up API connection** cell.

### Download sample audio

In [ ]:
import urllib.request

# Sample English speech audio (WAV, MIT License)
# Source: https://github.com/Azure-Samples/cognitive-services-speech-sdk
audio_path = "/content/sample_speech.wav"

urllib.request.urlretrieve(
    "https://github.com/Azure-Samples/cognitive-services-speech-sdk/raw/master/sampledata/audiofiles/katiesteve.wav",
    audio_path,
)
print(f"✅ Sample downloaded → {audio_path}")

### (Optional) Upload your own audio file instead

Run this cell manually to upload your own file. It will override the sample above.

In [ ]:
# NOTE: Run this cell manually only — it is intentionally skipped during 'Run all'.
# Uncomment the lines below and run this cell to upload your own audio file.

# from google.colab import files as colab_files
# print("Select an audio file to upload (mp3, wav, m4a, ogg, flac, webm, …)")
# uploaded = colab_files.upload()
# if uploaded:
#     audio_path = f"/content/{list(uploaded.keys())[0]}"
#     print(f"\nUploaded: {audio_path}")
# else:
#     print("No file uploaded — keeping previous audio_path.")

### Set up API connection

Run this cell once after choosing an audio file above.

In [ ]:
import os, requests

assert audio_path and os.path.exists(audio_path), \
    "No audio file found — run the sample download cell or upload cell above first."

REST_BASE = f"http://127.0.0.1:{WHISPERLIVE_REST_PORT}"
WS_URL    = f"ws://127.0.0.1:{WHISPERLIVE_PORT}"
# Use Bearer headers for both APIs so the key never appears in the URL.
req_headers = {}
if WHISPERLIVE_API_KEY:
    req_headers["Authorization"] = f"Bearer {WHISPERLIVE_API_KEY}"

print(f"Audio file : {audio_path}")
print(f"REST API   : {REST_BASE}")
print(f"WebSocket  : {WS_URL}")
print('✅ API connection ready.')

## Step 5 — Transcribe via REST API

### Transcribe — JSON response (default)

In [ ]:
print("(The model will be downloaded from HuggingFace on first request — this may take a minute.)")

# whisper-1 (or omitting model) uses the model configured on the server.
rest_data = {"model": "whisper-1"}
if WHISPERLIVE_LANGUAGE != "auto":
    rest_data["language"] = WHISPERLIVE_LANGUAGE

with requests.Session() as session, open(audio_path, "rb") as f:
    session.trust_env = False  # Keep local audio and credentials out of environment proxies.
    response = session.post(
        f"{REST_BASE}/v1/audio/transcriptions",
        files={"file": (os.path.basename(audio_path), f)},
        data=rest_data,
        headers=req_headers,
        timeout=(10, 600),  # Allow time for the initial model download and CPU inference.
    )

response.raise_for_status()
print("📝 Transcript:")
print(response.json()["text"])
print('✅ Done.')

## Step 6 — Real-time streaming via WebSocket

The cell below streams the audio file over a WebSocket connection and prints transcription segments as they are decoded — the same experience as live microphone streaming, but using a pre-recorded file.

In [ ]:
import asyncio, json, uuid, numpy as np, soundfile as sf, websockets

# Audio parameters expected by WhisperLive (faster_whisper backend)
_SAMPLE_RATE  = 16000   # Hz
_CHUNK_FRAMES = 4096    # samples per WebSocket message (~256 ms at 16 kHz)

async def stream_file(audio_file: str, ws_url: str, model: str, language: str, headers: dict) -> None:
    """
    Connect to a WhisperLive WebSocket server, stream audio from a local file
    as float32 PCM chunks at 16 kHz, and print transcription segments as they
    arrive.
    """
    # Load audio and resample to 16 kHz mono float32
    audio, sr = sf.read(audio_file, dtype="float32", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)  # downmix to mono
    if sr != _SAMPLE_RATE:
        # Simple linear resampling; scipy is available on Colab if higher quality is needed
        target_len = int(len(audio) * _SAMPLE_RATE / sr)
        audio = np.interp(
            np.linspace(0, len(audio) - 1, target_len),
            np.arange(len(audio)),
            audio,
        ).astype(np.float32)

    lang_param = None if language == "auto" else language
    client_uid = str(uuid.uuid4())

    async with websockets.connect(ws_url, additional_headers=headers, proxy=None) as ws:
        # 1. Send connection handshake
        await ws.send(json.dumps({
            "uid":      client_uid,
            "language": lang_param,
            "task":     "transcribe",
            "model":    model,
            "use_vad":  True,
        }))

        # Wait for the model to load before streaming any audio.
        print("Waiting for the server model to be ready…")
        ready_deadline = asyncio.get_running_loop().time() + 600
        while True:
            remaining = max(0, ready_deadline - asyncio.get_running_loop().time())
            try:
                data = json.loads(await asyncio.wait_for(ws.recv(), timeout=remaining))
            except TimeoutError as exc:
                raise TimeoutError("Timed out waiting for the server model. Check server logs.") from exc
            if data.get("uid") != client_uid:
                continue
            if data.get("status") in {"ERROR", "WAIT"}:
                raise RuntimeError(f"WhisperLive could not start streaming: {data.get('message')}")
            if data.get("message") == "SERVER_READY":
                break

        print(f"✅ Connected (uid={client_uid[:8]}…), streaming {len(audio)/_SAMPLE_RATE:.1f}s of audio\n")

        # 2. Stream audio chunks and receive results concurrently
        full_transcript = ""
        printed_segs = set()
        last_incomplete = [None]  # mutable container to allow update inside nested async def

        async def send_audio():
            for start in range(0, len(audio), _CHUNK_FRAMES):
                chunk = audio[start : start + _CHUNK_FRAMES]
                await ws.send(chunk.tobytes())
                await asyncio.sleep(_CHUNK_FRAMES / _SAMPLE_RATE * 0.9)  # ~real-time pacing
            await ws.send(b"END_OF_AUDIO")

        async def receive_segments():
            nonlocal full_transcript
            async for raw in ws:
                try:
                    data = json.loads(raw)
                except Exception:
                    continue
                if data.get("uid") != client_uid:
                    continue
                if data.get("status") in {"ERROR", "WAIT"}:
                    raise RuntimeError(f"WhisperLive streaming failed: {data.get('message')}")
                if "segments" in data:
                    last_incomplete[0] = None  # Discard partials superseded by this update.
                for seg in data.get("segments", []):
                    text = seg.get("text", "").strip()
                    completed = seg.get("completed", False)
                    seg_key = (seg.get("start"), seg.get("end"), text)
                    if text and completed and seg_key not in printed_segs:
                        printed_segs.add(seg_key)
                        start_t = float(seg.get("start", 0.0))
                        end_t   = float(seg.get("end", 0.0))
                        print(f"[{start_t:.2f}s → {end_t:.2f}s]  {text}")
                        full_transcript += " " + text
                    elif text and not completed:
                        last_incomplete[0] = seg  # track the latest incomplete segment
                if data.get("message") == "DISCONNECT":
                    break

        tasks = [asyncio.create_task(send_audio()), asyncio.create_task(receive_segments())]
        try:
            await asyncio.gather(*tasks)
        finally:
            for task in tasks:
                task.cancel()
            await asyncio.gather(*tasks, return_exceptions=True)

        # Print any remaining partial segment that was not marked complete.
        if last_incomplete[0]:
            text = last_incomplete[0].get("text", "").strip()
            seg_key = (last_incomplete[0].get("start"), last_incomplete[0].get("end"), text)
            if text and seg_key not in printed_segs:
                start_t = float(last_incomplete[0].get("start", 0.0))
                end_t   = float(last_incomplete[0].get("end", 0.0))
                print(f"[{start_t:.2f}s → {end_t:.2f}s]  {text}")
                full_transcript += " " + text

        if full_transcript.strip():
            print(f"\n✅ Full transcript: {full_transcript.strip()}")


await stream_file(
    audio_file=audio_path,
    ws_url=WS_URL,
    model=WHISPERLIVE_MODEL,
    language=WHISPERLIVE_LANGUAGE,
    headers=req_headers,
)
print('✅ Done.')

## View server logs

In [ ]:
!tail -40 /content/whisper_live_server.log

## Stop the server

In [ ]:
if "server_proc" in dir() and server_proc.poll() is None:
    stop_server()
    print("✅ Server stopped.")
else:
    print("ℹ️  Server is not running.")